In [ ]:

# Install tensorflow if not already installed
try:
    import tensorflow as tf
except ImportError:
    print("Installing TensorFlow...")
    !pip install tensorflow
    import tensorflow as tf
    print("TensorFlow installed and imported successfully.")

# Install openpyxl if not already installed (required for pandas to read .xlsx files)
try:
    import openpyxl
except ImportError:
    print("Installing openpyxl...")
    !pip install openpyxl
    print("openpyxl installed successfully.")


import os
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.cm as cm

from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score

from tensorflow.keras import Input
from tensorflow.keras.models import Sequential, Model
from tensorflow.keras.layers import Dense
from tensorflow.keras.optimizers import SGD

# ==============================================================================
# 0. SETTINGS
# ==============================================================================

DATA_PATH = "marketing_campaign.xlsx"   # path to the dataset used by all notebooks
RESULTS_ACC_CSV = "epoch_val_accuracy_all_models.csv"  # per-epoch VALIDATION accuracy
PLOT_ACC_PATH = "epoch_val_accuracy_all_22_models.png"

# Full epoch grid used by every original notebook.
EPOCH_LIST_FULL = [
    50, 100, 150, 200, 250, 300, 350, 400, 450, 500,
    600, 700, 800, 900, 1000, 1200, 1400, 1600, 1800, 2000,
]

# Set QUICK_TEST=True to sanity-check the whole pipeline quickly (small nets,
# few epochs) before committing to the full multi-hour run.
QUICK_TEST = False
EPOCH_LIST = [5, 10, 20] if QUICK_TEST else EPOCH_LIST_FULL

RANDOM_STATE = 42

# Fraction of the ORIGINAL dataset held out as the untouched final test set.
TEST_SIZE = 0.20
# Fraction of the REMAINING (non-test) data held out as the validation set,
# used for every epoch-sweep metric and the final plot.
VAL_SIZE = 0.20


# ==============================================================================
# 1. LOAD + CLEAN + FEATURE-ENGINEER THE DATASET
#    (identical preprocessing pipeline used in every one of the 22 notebooks,
#    now producing TRAIN / VALIDATION / TEST instead of just TRAIN / TEST)
# ==============================================================================

def load_and_prepare_data(path=DATA_PATH):
    if not os.path.exists(path):
        # Fall back to the Google Colab upload widget if running in Colab and
        # the file isn't already present locally.
        try:
            from google.colab import files  # noqa
            print(f"'{path}' not found locally - please upload it now.")
            uploaded = files.upload()
            path = list(uploaded.keys())[0]
        except ImportError:
            raise FileNotFoundError(
                f"Could not find '{path}'. Place marketing_campaign.xlsx next "
                f"to this script (or run in Colab to be prompted to upload it)."
            )

    df = pd.read_excel(path)

    # Missing values
    df["Income"] = df["Income"].fillna(df["Income"].median())

    # Duplicates
    df = df.drop_duplicates()

    # Feature engineering
    df["Dt_Customer"] = pd.to_datetime(df["Dt_Customer"])
    latest_date = df["Dt_Customer"].max()
    df["Customer_Tenure"] = (latest_date - df["Dt_Customer"]).dt.days
    df["Age"] = 2026 - df["Year_Birth"]
    df = df[df["Age"] <= 100]
    df["Total_Children"] = df["Kidhome"] + df["Teenhome"]
    df["Total_Spending"] = (
        df["MntWines"] + df["MntFruits"] + df["MntMeatProducts"]
        + df["MntFishProducts"] + df["MntSweetProducts"] + df["MntGoldProds"]
    )
    df["Total_Purchases"] = (
        df["NumWebPurchases"] + df["NumCatalogPurchases"] + df["NumStorePurchases"]
    )

    # Drop irrelevant columns
    df = df.drop(columns=["ID", "Dt_Customer", "Z_CostContact", "Z_Revenue"])

    # Encode categoricals
    le = LabelEncoder()
    df["Education"] = le.fit_transform(df["Education"])
    df["Marital_Status"] = le.fit_transform(df["Marital_Status"])

    # Features / target
    X = df.drop("Response", axis=1)
    y = df["Response"]

    # --- Step 1: split off the TEST set first. This set is untouched after
    #     this point - it is not scaled-and-used, not predicted on, not
    #     plotted, anywhere else in this script. ---
    X_trainval, X_test, y_trainval, y_test = train_test_split(
        X, y, test_size=TEST_SIZE, random_state=RANDOM_STATE, stratify=y
    )

    # --- Step 2: split the remaining trainval data into TRAIN and
    #     VALIDATION. Every epoch-sweep metric and the final plot use the
    #     validation set. ---
    X_train, X_val, y_train, y_val = train_test_split(
        X_trainval, y_trainval, test_size=VAL_SIZE,
        random_state=RANDOM_STATE, stratify=y_trainval
    )

    # --- Scaling: fit ONLY on the training data, then transform train/val/test.
    #     (X_test_scaled is computed here so it exists if you want it later,
    #     but this script never evaluates anything on it.) ---
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_val_scaled = scaler.transform(X_val)
    X_test_scaled = scaler.transform(X_test)

    print(f"Split sizes -> train: {X_train_scaled.shape[0]}, "
          f"validation: {X_val_scaled.shape[0]}, "
          f"test (held out, untouched): {X_test_scaled.shape[0]}")

    return (X_train_scaled, X_val_scaled, X_test_scaled,
            y_train, y_val, y_test)


# ==============================================================================
# 2. CUSTOM "EXPONENTIAL" SVM KERNEL
#    (identical definition used in every EXPONENTIAL-kernel notebook)
# ==============================================================================

def exponential_kernel(X, Y):
    gamma = 0.01
    K = np.zeros((X.shape[0], Y.shape[0]))
    for i in range(X.shape[0]):
        for j in range(Y.shape[0]):
            distance = np.linalg.norm(X[i] - Y[j])
            K[i, j] = np.exp(-gamma * distance)
    return K


# ==============================================================================
# 3. THE 22 MODEL CONFIGURATIONS
#    Extracted directly from the 22 source notebooks:
#      - hidden_sizes: sizes of the ANN's hidden Dense(relu) layers, in order
#      - feature_layer_index: which of those hidden layers' activations are
#        fed into the SVM (0 = first hidden layer, 1 = second, ...)
#      - kernel: 'rbf' or 'exponential'
#      - label: legend label (matches the original notebooks' plot labels)
# ==============================================================================

MODEL_CONFIGS = [
    {"hidden_sizes": [100, 50],          "feature_layer_index": 0, "kernel": "rbf",         "label": "Input-100-50-SVM | Hidden Layer 1 | RBF"},
    {"hidden_sizes": [100, 50],          "feature_layer_index": 1, "kernel": "rbf",         "label": "Input-100-50-SVM | Hidden Layer 2 | RBF"},
    {"hidden_sizes": [500, 100],         "feature_layer_index": 0, "kernel": "rbf",         "label": "Input-500-100-SVM | Hidden Layer 1 | RBF"},
    {"hidden_sizes": [500, 100],         "feature_layer_index": 1, "kernel": "rbf",         "label": "Input-500-100-SVM | Hidden Layer 2 | RBF"},
    {"hidden_sizes": [2000, 1000, 100],  "feature_layer_index": 2, "kernel": "exponential", "label": "Input-2000-1000-100-SVM | Hidden Layer 3 | EXPONENTIAL"},
    {"hidden_sizes": [2000, 1000, 100],  "feature_layer_index": 2, "kernel": "rbf",         "label": "Input-2000-1000-100-SVM | Hidden Layer 3 | RBF"},
    {"hidden_sizes": [1000, 500, 50],    "feature_layer_index": 0, "kernel": "exponential", "label": "Input-1000-500-50-SVM | Hidden Layer 1 | EXPONENTIAL"},
    {"hidden_sizes": [2000, 1000, 100],  "feature_layer_index": 1, "kernel": "exponential", "label": "Input-2000-1000-100-SVM | Hidden Layer 2 | EXPONENTIAL"},
    {"hidden_sizes": [2000, 1000, 50],   "feature_layer_index": 1, "kernel": "exponential", "label": "Input-2000-1000-50-SVM | Hidden Layer 2 | EXPONENTIAL"},
    {"hidden_sizes": [1000, 500, 50],    "feature_layer_index": 0, "kernel": "rbf",         "label": "Input-1000-500-50-SVM | Hidden Layer 1 | RBF"},
    {"hidden_sizes": [2000, 1000, 100],  "feature_layer_index": 1, "kernel": "rbf",         "label": "Input-2000-1000-100-SVM | Hidden Layer 2 | RBF"},
    {"hidden_sizes": [2000, 1000, 50],   "feature_layer_index": 1, "kernel": "rbf",         "label": "Input-2000-1000-50-SVM | Hidden Layer 2 | RBF"},
    {"hidden_sizes": [2000, 1000, 100],  "feature_layer_index": 0, "kernel": "exponential", "label": "Input-2000-1000-100-SVM | Hidden Layer 1 | EXPONENTIAL"},
    {"hidden_sizes": [2000, 1000, 50],   "feature_layer_index": 0, "kernel": "exponential", "label": "Input-2000-1000-50-SVM | Hidden Layer 1 | EXPONENTIAL"},
    {"hidden_sizes": [2000, 1000, 100],  "feature_layer_index": 0, "kernel": "rbf",         "label": "Input-2000-1000-100-SVM | Hidden Layer 1 | RBF"},
    {"hidden_sizes": [2000, 1000, 50],   "feature_layer_index": 0, "kernel": "rbf",         "label": "Input-2000-1000-50-SVM | Hidden Layer 1 | RBF"},
    {"hidden_sizes": [1000, 500, 50],    "feature_layer_index": 2, "kernel": "exponential", "label": "Input-1000-500-50-SVM | Hidden Layer 3 | EXPONENTIAL"},
    {"hidden_sizes": [2000, 1000, 50],   "feature_layer_index": 2, "kernel": "exponential", "label": "Input-2000-1000-50-SVM | Hidden Layer 3 | EXPONENTIAL"},
    {"hidden_sizes": [1000, 500, 50],    "feature_layer_index": 2, "kernel": "rbf",         "label": "Input-1000-500-50-SVM | Hidden Layer 3 | RBF"},
    {"hidden_sizes": [2000, 1000, 50],   "feature_layer_index": 2, "kernel": "rbf",         "label": "Input-2000-1000-50-SVM | Hidden Layer 3 | RBF"},
    {"hidden_sizes": [1000, 500, 50],    "feature_layer_index": 1, "kernel": "exponential", "label": "Input-1000-500-50-SVM | Hidden Layer 2 | EXPONENTIAL"},
    {"hidden_sizes": [1000, 500, 50],    "feature_layer_index": 1, "kernel": "rbf",         "label": "Input-1000-500-50-SVM | Hidden Layer 2 | RBF"},
]

assert len(MODEL_CONFIGS) == 22
assert len(set(c["label"] for c in MODEL_CONFIGS)) == 22  # all labels unique


# ==============================================================================
# 4. BUILD ANN, EXTRACT FEATURES, TRAIN SVM, RETURN ACCURACY - FOR ONE
#    (model, epoch), scored on the VALIDATION set. The TEST set is not passed
#    into this function at all, so it cannot be touched here.
# ==============================================================================

def train_and_score(config, epoch_value, X_train_scaled, X_val_scaled, y_train, y_val):
    hidden_sizes = config["hidden_sizes"]
    feat_idx = config["feature_layer_index"]
    kernel = config["kernel"]

    # --- Build ANN, naming the layer that feeds the SVM 'feature_layer' ---
    layers = [Input(shape=(X_train_scaled.shape[1],))]
    for i, units in enumerate(hidden_sizes):
        name = "feature_layer" if i == feat_idx else None
        layers.append(Dense(units, activation="relu", name=name))
    layers.append(Dense(1, activation="sigmoid"))
    ann = Sequential(layers)

    ann.compile(
        optimizer=SGD(learning_rate=0.01),
        loss="binary_crossentropy",
        metrics=["accuracy"],
    )

    ann.fit(
        X_train_scaled, y_train,
        epochs=epoch_value,
        batch_size=32,
        verbose=0,
    )

    # --- Extract activations of the chosen hidden layer ---
    feature_model = Model(inputs=ann.inputs, outputs=ann.get_layer("feature_layer").output)
    X_train_features = feature_model.predict(X_train_scaled, verbose=0)
    X_val_features = feature_model.predict(X_val_scaled, verbose=0)

    # --- Train the SVM classifier on the TRAIN features only ---
    if kernel == "exponential":
        svm_model = SVC(kernel=exponential_kernel, probability=True, random_state=RANDOM_STATE)
    else:
        svm_model = SVC(kernel="rbf", C=1, gamma="scale", probability=True, random_state=RANDOM_STATE)

    svm_model.fit(X_train_features, y_train)

    # --- Evaluate on the VALIDATION set only ---
    y_val_pred = svm_model.predict(X_val_features)
    val_acc = accuracy_score(y_val, y_val_pred)

    return val_acc


# ==============================================================================
# 5. RUN THE FULL EXPERIMENT: 22 MODELS x len(EPOCH_LIST) EPOCH CHECKPOINTS
#    All scoring happens on the VALIDATION set. X_test_scaled/y_test are not
#    passed in, so this function has no way to touch the test set.
# ==============================================================================

def run_all_models(X_train_scaled, X_val_scaled, y_train, y_val):
    all_acc_results = {}  # label -> list of validation accuracy scores

    total_runs = len(MODEL_CONFIGS) * len(EPOCH_LIST)
    run_counter = 0
    t_start = time.time()

    for m_idx, config in enumerate(MODEL_CONFIGS, start=1):
        label = config["label"]
        acc_scores = []
        print(f"\n=== [{m_idx}/{len(MODEL_CONFIGS)}] {label} ===")

        for epoch_value in EPOCH_LIST:
            run_counter += 1
            val_acc = train_and_score(
                config, epoch_value, X_train_scaled, X_val_scaled, y_train, y_val
            )
            acc_scores.append(val_acc)
            elapsed = time.time() - t_start
            print(f"  Epoch={epoch_value:<5} Val Acc={val_acc:.4f}   "
                  f"[{run_counter}/{total_runs} runs, {elapsed/60:.1f} min elapsed]")

        all_acc_results[label] = acc_scores
        best_i = int(np.argmax(acc_scores))
        print(f"  -> Best Epoch (by Val Accuracy) = {EPOCH_LIST[best_i]} | "
              f"Best Val Accuracy = {acc_scores[best_i]:.4f}")

    return all_acc_results


# ==============================================================================
# 6. PLOT: ALL 22 MODELS, VALIDATION ACCURACY VS. EPOCH, WITH A LEGEND
# ==============================================================================

def _model_color_list(n_models):
    """22-color palette used for the accuracy plot."""
    colors = cm.get_cmap("tab20", n_models)
    extra_colors = cm.get_cmap("Set2", 2)  # a couple more for indices 20-21
    return [colors(i) for i in range(20)] + [extra_colors(0), extra_colors(1)]


def plot_all_models_accuracy(all_acc_results):
    plt.figure(figsize=(16, 10))
    color_list = _model_color_list(len(all_acc_results))

    for i, (label, acc_scores) in enumerate(all_acc_results.items()):
        plt.plot(
            EPOCH_LIST,
            np.array(acc_scores) * 100,
            marker="o",
            markersize=3,
            linewidth=1.5,
            color=color_list[i % len(color_list)],
            label=label,
        )

    plt.xlabel("Epoch")
    plt.ylabel("Validation Accuracy (%)")
    plt.title("Epoch Optimization using Validation Accuracy — All 22 Models")
    plt.ylim(0, 100)
    plt.yticks(np.arange(0, 101, 10))
    plt.grid(True, alpha=0.3)

    # Legend outside the plot area (22 entries need the extra room)
    plt.legend(
        loc="upper left",
        bbox_to_anchor=(1.01, 1.0),
        fontsize=8,
        ncol=1,
        frameon=True,
    )
    plt.tight_layout()
    plt.savefig(PLOT_ACC_PATH, dpi=150, bbox_inches="tight")
    plt.show()
    print(f"Saved accuracy plot to: {PLOT_ACC_PATH}")


# ==============================================================================
# 7. MAIN
# ==============================================================================

if __name__ == "__main__":
    print("Loading and preparing data (train / validation / test split)...")
    (X_train_scaled, X_val_scaled, X_test_scaled,
     y_train, y_val, y_test) = load_and_prepare_data()
    print(f"Train shape: {X_train_scaled.shape}, "
          f"Validation shape: {X_val_scaled.shape}, "
          f"Test shape: {X_test_scaled.shape} (held out, not used below)")

    print(f"\nRunning {len(MODEL_CONFIGS)} models x {len(EPOCH_LIST)} epoch "
          f"checkpoints = {len(MODEL_CONFIGS) * len(EPOCH_LIST)} total training runs, "
          f"scored on the VALIDATION set.")
    if QUICK_TEST:
        print(">>> QUICK_TEST is ON: using a small epoch grid for a fast sanity check. <<<")

    # NOTE: X_test_scaled / y_test are intentionally NOT passed into
    # run_all_models - the test set is never touched below this point.
    all_acc_results = run_all_models(
        X_train_scaled, X_val_scaled, y_train, y_val
    )

    # Save raw validation results
    acc_df = pd.DataFrame(all_acc_results, index=EPOCH_LIST)
    acc_df.index.name = "epoch"
    acc_df.to_csv(RESULTS_ACC_CSV)
    print(f"\nSaved per-epoch validation accuracy scores to: {RESULTS_ACC_CSV}")

    # Summary table of best epoch per model (by validation accuracy)
    summary_rows = []
    for label, acc_scores in all_acc_results.items():
        best_i = int(np.argmax(acc_scores))
        summary_rows.append({
            "model": label,
            "best_epoch": EPOCH_LIST[best_i],
            "best_val_accuracy": acc_scores[best_i],
        })
    summary_df = pd.DataFrame(summary_rows).sort_values("best_val_accuracy", ascending=False)
    print("\n=== BEST EPOCH / VALIDATION ACCURACY PER MODEL "
          "(sorted by best validation accuracy) ===")
    print(summary_df.to_string(index=False))

    # Plot (validation accuracy vs epoch, all 22 models)
    plot_all_models_accuracy(all_acc_results)


